In [1]:
# FIAT calibration workflow
import fiatmodel

# `xarray` to read the observation NetCDF file
import xarray as xr

In [2]:
# defining MESH calibration parameter bounds
class_dict_bounds = {
    6: [
        {
            'class': 'needleleaf',
            'lnz0': [0.5, 4.0],
        },
        {
            'class': 'broadleaf',
            'lnz0': [0.2, 0.8]
        },
    ]
}

hydrology_dict_bounds = {
    15: {
        'zpls': [0.02, 0.6],
    },
}

routing_dict_bounds = {
    6: {
        'flz': [1e-6, 1e-3, "log10"],
    },
}

In [3]:
# Reading observed values
obs_obj = xr.open_dataset('./wolf-creek-research-basin/wolf-creek-gauge-data.nc')

# Creating a `Calibration` object
c = fiatmodel.Calibration(
    calibration_software = 'ostrich',
    model_software = 'mesh',
    calibration_config = {
        'instance_path': './wolf-creek-research-basin-calibration/',
        'random_seed': 10, # could be any integer!
        'algorithm': 'DDS', # Ostrich-specific algorithm keyword
        'algorithm_specs': { # refer to ostrich software manual for these keys
            'PerturbationValue': 0.2,
            'MaxIteration': 10_000,
            'UseRandomParamValue': None,
        },
        'spinup_start': '1980-01-01 12:00:00', # spin-up start date
        'dates': [ # one or more calibration dates, here only one
            {
                'start': '1980-01-01 23:00:00',
                'end': '1980-01-02 23:00:00',
            },
        ],
        'objective_functions': {
            # '_helpers': {
            #     'QO': {
            #         'kge_2012': ['-1 * alaska_72', '-2 * alaska_72'],
            #         'kge_all': 'sum(kge_2012) / (2 - len(kge_2012))',
            #     },
            # },
            # 'custom': {
            #     'QO': {
            #         'kge_prime': 'kge_all / 2 - kge_all'
            #     },
            # },
            'fluxes': { # can also include state variables
                'QO': {
                    'kge_2012': '-1 * alaska_72', # KGE-2012 value calculated using `alaska_72` gauge data
                },
            },
        },
    },
    model_config = {
        'instance_path': './wolf-creek-research-basin/',
        'parameter_bounds': {
            'class': class_dict_bounds,
            'hydrology': hydrology_dict_bounds,
            'routing': routing_dict_bounds,
        },
        'executable': 'sa_mesh', # to be added to the copying stuff + required_files
    },
    observations = [
        {
            "name": "alaska_72",
            "type": "QO", # the output of MESH to be compared with
            "timeseries": obs_obj['discharge'].isel(gauge_name=2).to_series(), # have a look at `obs_obj` separately yourself!
            "unit": "m^3/s", # unit of the observed values
            "scale_factor": 1,
            "offset_factor": 0,
            "computational_unit": "subbasin", # dimension of computational unit in MESH
            "computational_unit_id": 38, # this is up to user to specify the accurate position of observed values
            "freq": "1h", # frequency of the observed values, necessary due to potential missing values with observation records
        },
    ],
)

In [4]:
c.prepare(output_path='./wolf-creek-research-basin-calibration/')

/Users/kasrakeshavarz/Documents/github-repos/fiatmodel/src/fiatmodel/models/mesh/model.py:613: UserWarning: The routing section in MESH_parameters_hydrology.ini is empty. Reading `MESH_parameters.nc` file.
  warnings.warn(f"The routing section in MESH_parameters_hydrology.ini"
/Users/kasrakeshavarz/Documents/github-repos/fiatmodel/src/fiatmodel/calibration/optimizer.py:309: UserWarning: The directory ./wolf-creek-research-basin-calibration/ already exists. Contents may be overwritten.
  warnings.warn(f"The directory {path} already exists."
/Users/kasrakeshavarz/Documents/github-repos/fiatmodel/src/fiatmodel/calibration/optimizer.py:309: UserWarning: The directory ./wolf-creek-research-basin-calibration/etc/templates already exists. Contents may be overwritten.
  warnings.warn(f"The directory {path} already exists."
/Users/kasrakeshavarz/Documents/github-repos/fiatmodel/src/fiatmodel/calibration/optimizer.py:309: UserWarning: The directory ./wolf-creek-research-basin-calibration/etc alr